# 07 · De puntos medidos a mapas: interpolación espacial

## Objetivos y recorrido

- Distinguir mediciones puntuales y predicciones sobre una malla.
- Usar coordenadas en metros y definir un dominio de representación.
- Comparar vecino más cercano, IDW, una tendencia plana y kriging ordinario.
- Dibujar mapas comparables con puntos, escala de color y coordenadas.
- Evaluar predicciones en puntos reservados y discutir incertidumbre.

Comenzaremos con mediciones reales de suelo. El recorrido principal es puntos → malla → métodos → mapas; el variograma y la validación requieren una lectura más pausada con el docente.


## Qué se puede interpolar

Interpolar es estimar una variable en ubicaciones sin medición a partir de puntos próximos y de supuestos sobre su continuidad espacial. Extrapolar implica predecir fuera del área representada por las observaciones.

Una concentración en suelo, una temperatura medida simultáneamente en estaciones o una elevación son ejemplos posibles. Los códigos de litología, especie y hábitat no son cantidades continuas. Tampoco se puede crear un mapa espacial a partir de la serie meteorológica de una sola coordenada del notebook 04.

Cada predicción debe conservar la unidad y el soporte de las mediciones. Un punto de muestreo puede representar una muestra compuesta, no un volumen infinitesimal.


## Datos y paquetes

Usaremos `meuse`, disponible en el paquete `sp`: mediciones de metales en muestras compuestas de suelo de una llanura aluvial cerca de Stein, Países Bajos. El zinc está en mg/kg y las coordenadas Este–Norte están en metros, en el sistema neerlandés RD. [Descripción, autores y referencia de los datos](https://edzer.github.io/sp/reference/meuse.html).

Este contexto conecta geoquímica del suelo y condiciones ambientales relevantes para Biología. No son datos de Colombia ni datos sintéticos del curso.

Se necesitan `sp` (datos), `sf` (geometrías), `gstat` (interpolación) y `ggplot2` (figuras). Si faltan, instala una vez en una celda aparte:

```r
install.packages(c("sp", "sf", "gstat", "ggplot2"), repos = "https://cloud.r-project.org")
```

En Colab la instalación puede tardar; en Linux `sf` necesita sus bibliotecas de sistema. Carga los paquetes antes de continuar. No se requieren un DEM ni mapas de fondo descargados.


### ¿Qué es `sf`?

`sf` significa *simple features*: es un paquete de R para representar objetos espaciales como puntos, líneas y polígonos. Un objeto `sf` se parece a un `data.frame`, pero incluye una columna especial, `geometry`, que guarda la forma y ubicación de cada objeto. Además registra el sistema de referencia de coordenadas (**CRS**).

En nuestra tabla, el zinc es un **atributo** de la muestra y el punto donde se midió es su **geometría**. Convertirla a `sf` permite preguntar si un punto está dentro de un área o calcular una envolvente. Muchas funciones del paquete empiezan por `st_`; explicaremos cada una antes de usarla.

No hay que confundir los paquetes `sp` y `sf`: aquí `sp` aporta el conjunto `meuse`, y `sf` organiza las geometrías con las que trabajaremos. [Introducción oficial a sf](https://r-spatial.github.io/sf/articles/sf1.html).


In [ ]:
# install.packages(c("sp", "sf", "gstat", "ggplot2"), repos = "https://cloud.r-project.org")

In [ ]:
library(sf)
library(gstat)
library(ggplot2)
data("meuse", package = "sp")
head(meuse[, c("x", "y", "zinc")])


## Inspeccionar y preparar las mediciones

Conservamos las filas con coordenadas y zinc finitos, y zinc positivo para poder usar logaritmos. Revisamos coordenadas duplicadas: dos mediciones en el mismo lugar requieren una decisión previa; aquí no las promediaremos automáticamente.

La concentración es asimétrica. Trabajaremos con `log(zinc)` para comparar todos los métodos en la misma escala; `log()` es el logaritmo natural. Al volver con `exp()`, una predicción media en escala logarítmica no se convierte sin más en una media en mg/kg: el mapa se interpreta como una estimación central en escala multiplicativa.


In [ ]:
datos <- meuse[, c("x", "y", "zinc")]
datos <- datos[is.finite(datos$x) & is.finite(datos$y) &
                 is.finite(datos$zinc) & datos$zinc > 0, ]
sum(duplicated(datos[, c("x", "y")]))
stopifnot(!any(duplicated(datos[, c("x", "y")])))
datos$log_zinc <- log(datos$zinc)
summary(datos)


### De una tabla de coordenadas a una tabla espacial

Antes de calcular distancias, debemos saber qué representan las coordenadas y en qué unidad están. Estas mediciones ya tienen coordenadas proyectadas en metros, en RD New.

| Función de `sf` | Qué hace aquí |
|---|---|
| `st_as_sf(datos, coords = c("x", "y"), crs = 28992)` | Convierte cada fila en un punto usando `x` como Este e `y` como Norte, y declara su CRS. |
| `st_crs(puntos)` | Consulta el CRS. Su campo `$units_gdal` permite comprobar la unidad de las coordenadas. |
| `st_bbox(puntos)` | Devuelve el rectángulo que contiene todos los puntos: `xmin`, `ymin`, `xmax`, `ymax`. |

`remove = FALSE` conserva las columnas originales `x` e `y`, además de crear `geometry`. Así podemos seguir usándolas en cálculos y gráficos.

**Declarar un CRS no transforma las coordenadas.** `crs = 28992` describe estos datos; no debe asignarse a mediciones de otro lugar. Si tus datos están en longitud y latitud, identifica primero su CRS y usa `st_transform(objeto, crs = codigo_destino)` para convertirlos a un sistema proyectado apropiado para la región. No basta con cambiar la etiqueta del CRS para obtener metros.


In [ ]:
puntos <- st_as_sf(datos, coords = c("x", "y"), crs = 28992, remove = FALSE)
class(puntos)
head(puntos[, c("zinc", "geometry")])
st_crs(puntos)$units_gdal
st_bbox(puntos)


In [ ]:
# Ejemplo: df_original con longitud y latitud en WGS84 (EPSG:4326).
# Si el CRS de origen es otro, reemplaza 4326 por su c?digo correcto.
# Convertir a MAGNA-SIRGAS / Origen Nacional (EPSG:9377), en metros:
# puntos_geo <- st_as_sf(df_original, coords = c("longitud", "latitud"),
#                        crs = 4326, remove = FALSE)
# puntos_planos <- st_transform(puntos_geo, crs = 9377)
# xy <- st_coordinates(puntos_planos)  # Extrae las coordenadas transformadas.
# df_planas <- st_drop_geometry(puntos_planos)
# df_planas$x <- xy[, 1]  # Este (m)
# df_planas$y <- xy[, 2]  # Norte (m)
# df_planas conserva las columnas originales y a?ade x e y en metros.


In [ ]:
ggplot(datos, aes(x = x, y = y, color = zinc)) +
  geom_point(size = 2) +
  coord_equal() +
  scale_color_viridis_c(name = "Zn medido\n(mg/kg)") +
  labs(title = "Muestras de suelo · Meuse", x = "Este RD (m)", y = "Norte RD (m)") +
  theme_minimal()


## Crear una malla: dónde queremos predecir

Las muestras son las ubicaciones **medidas**. La malla es otro conjunto de puntos, organizado regularmente, donde queremos **estimar** zinc. Todavía no tiene concentraciones: las añadirá cada método de interpolación.

Para construirla hay que decidir tres cosas:

| Decisión | Control en el código | Efecto |
|---|---|---|
| Separación entre centros | `paso_m` | Hace la malla más fina o más gruesa. |
| Rectángulo de trabajo | `xmin`, `xmax`, `ymin`, `ymax` | Cambia la extensión donde generamos centros. |
| Recorte espacial | `recortar_envolvente` | Conserva solo los centros dentro de la envolvente de las muestras, o todo el rectángulo. |

### 1. Resolución: más o menos fina

`paso_m = 100` coloca centros separados 100 m en cada eje. Un paso de **50 m** crea una malla más fina; **200 m**, una más gruesa. Para la misma extensión, reducir el paso a la mitad produce aproximadamente cuatro veces más centros, porque aumentan las posiciones tanto en X como en Y. También aumenta el tiempo de cálculo.

Esto cambia la separación de las predicciones y el tamaño de los cuadrados dibujados. **No añade nuevas mediciones ni garantiza más precisión.** En este ejercicio, `geom_tile()` representará cada centro con un cuadrado de lado `paso_m`.

### 2. Extensión: los límites del rectángulo

Partimos del rectángulo de las muestras, obtenido con `st_bbox()`. `margen_m` permite ampliarlo por todos los lados: con 0 se conservan los límites originales; con 300 se añaden 300 m hacia el oeste, este, sur y norte. `unname()` deja el número sin el nombre que tenía dentro de `caja`.


In [ ]:
# Controles de la malla: modifica aquí y vuelve a ejecutar desde esta celda.
paso_m <- 100                 # 50: más fina; 200: más gruesa
margen_m <- 0                 # Por ejemplo, 300 amplía el rectángulo
recortar_envolvente <- TRUE   # FALSE conserva todo el rectángulo

caja <- st_bbox(puntos)
xmin <- unname(caja["xmin"]) - margen_m
xmax <- unname(caja["xmax"]) + margen_m
ymin <- unname(caja["ymin"]) - margen_m
ymax <- unname(caja["ymax"]) + margen_m


Si quieres una zona concreta, **sobrescribe los cuatro límites después de la celda anterior y antes de generar los ejes**. Por ejemplo, este rectángulo cubre una parte del conjunto Meuse (todas las cifras están en metros RD):

```r
xmin <- 179000
xmax <- 180500
ymin <- 330000
ymax <- 332000
```

Para ampliar el área puedes aumentar `margen_m` o escribir otros límites. Sin embargo, con `recortar_envolvente = TRUE` seguiremos descartando los centros exteriores a la envolvente de las muestras: ampliar el rectángulo no desactiva ese recorte. Para conservar el rectángulo completo usa `FALSE`, reconociendo que fuera de la envolvente estarías extrapolando.

Cambiar la extensión selecciona destinos de predicción y la ventana de los mapas; las muestras usadas para ajustar los modelos siguen siendo las mismas. Para actualizar los resultados, vuelve a ejecutar **todas las celdas desde los controles de la malla hasta el final**.

### 3. Combinar los ejes: de secuencias a coordenadas

`seq(inicio, fin, by = paso_m)` genera posiciones en un eje. Si el intervalo no es múltiplo del paso, la última posición queda antes del límite máximo. `expand.grid(x = eje_x, y = eje_y)` construye todas las combinaciones de X e Y: con 3 posiciones en X y 4 en Y obtendríamos 12 centros.


In [ ]:
stopifnot(paso_m > 0, xmin < xmax, ymin < ymax)
extension <- list(x = c(xmin, xmax), y = c(ymin, ymax))
eje_x <- seq(xmin, xmax, by = paso_m)
eje_y <- seq(ymin, ymax, by = paso_m)
malla_rectangular <- expand.grid(x = eje_x, y = eje_y)
head(malla_rectangular)
c(posiciones_x = length(eje_x), posiciones_y = length(eje_y),
  centros_rectangulo = nrow(malla_rectangular))


Comprobemos el efecto del paso sin recalcular aún las interpolaciones. La siguiente tabla cuenta centros del **rectángulo**, antes del recorte. No modifica el valor de `paso_m` elegido arriba.


In [ ]:
comparacion_resolucion <- data.frame(paso_m = c(200, 100, 50), centros_rectangulo = 0L)
for (i in seq_len(nrow(comparacion_resolucion))) {
  paso_comparado <- comparacion_resolucion$paso_m[i]
  n_x <- length(seq(xmin, xmax, by = paso_comparado))
  n_y <- length(seq(ymin, ymax, by = paso_comparado))
  comparacion_resolucion$centros_rectangulo[i] <- n_x * n_y
}
comparacion_resolucion


### 4. Construir la envolvente de las muestras

`st_union(puntos)` reúne las geometrías en un único conjunto espacial. `st_convex_hull()` calcula su **envolvente convexa**: un polígono como una banda elástica alrededor de los puntos exteriores. Aquí actúa sobre el conjunto completo, no sobre cada punto por separado.

Este límite reduce la extrapolación exterior. Puede contener huecos sin muestras y no representa un límite ambiental oficial: estar dentro no garantiza buen apoyo de mediciones cercanas.


In [ ]:
conjunto_puntos <- st_union(puntos)
envolvente <- st_convex_hull(conjunto_puntos)


### 5. Convertir los centros a `sf` y aplicar el recorte

Volvemos a usar `st_as_sf()`, ahora con los centros de predicción. `st_crs(puntos)` asegura que malla y muestras comparten el mismo CRS.

`st_intersects(malla_sf, envolvente)` pregunta qué centros tocan o están dentro del polígono. Devuelve una lista con las coincidencias de cada centro. `lengths(...) > 0` la convierte en un vector lógico: `TRUE` si hay alguna coincidencia, `FALSE` si no hay ninguna. Este vector permite seleccionar las filas, como en las tablas de las clases anteriores.

Finalmente, `st_drop_geometry()` retira la columna espacial y devuelve una tabla ordinaria con X e Y. Conservamos los dos objetos: `malla_sf` para las operaciones espaciales y `malla` para guardar predicciones y dibujar. No se modifica la tabla de muestras.


In [ ]:
malla_sf <- st_as_sf(malla_rectangular, coords = c("x", "y"),
                     crs = st_crs(puntos), remove = FALSE)
dentro <- lengths(st_intersects(malla_sf, envolvente)) > 0
if (recortar_envolvente) {
  malla_sf <- malla_sf[dentro, ]
}
stopifnot(nrow(malla_sf) > 0)
malla <- st_drop_geometry(malla_sf)
c(mediciones = nrow(puntos), centros_rectangulo = nrow(malla_rectangular),
  centros_utilizados = nrow(malla))


### Mirar la malla antes de modelar

Dibujamos los centros del rectángulo en gris, los utilizados en azul y las muestras en naranja. El contorno es la envolvente. `plot()` reconoce la geometría de `sf` y puede añadir ese polígono a la figura abierta. Esta revisión permite detectar una extensión o un paso inadecuados antes de interpolar.


In [ ]:
plot(malla_rectangular$x, malla_rectangular$y, asp = 1, pch = 16, cex = 0.35,
     col = "grey80", xlim = extension$x, ylim = extension$y,
     xlab = "Este RD (m)", ylab = "Norte RD (m)", main = "Muestras y centros de predicción")
points(malla$x, malla$y, pch = 16, cex = 0.4, col = "steelblue")
plot(envolvente, add = TRUE, border = "grey30", lwd = 1.5)
points(datos$x, datos$y, pch = 16, cex = 0.6, col = "darkorange3")
legend("topleft", legend = c("Centros del rectángulo", "Centros utilizados", "Muestras"),
       col = c("grey80", "steelblue", "darkorange3"), pch = 16, cex = 0.8, bty = "n")


## Métodos que compararemos

| Método | Idea | Parámetro o supuesto |
|---|---|---|
| Vecino más cercano | Asignar el valor del punto más próximo. | Zonas constantes con cambios bruscos. |
| IDW | Promediar con más peso para puntos próximos. | Potencia `p` y número de vecinos. |
| Tendencia plana | Ajustar una superficie lineal a las coordenadas. | Variación suave de gran escala; no reproduce cada muestra. |
| Kriging ordinario | Ponderar usando dependencia espacial modelada. | Variograma y media local constante bajo el modelo. |

No hay un método universalmente mejor. Una superficie suave puede ocultar variación; una detallada puede reflejar la distribución de las muestras.


## 1. Vecino más cercano: copiar la medición más próxima

Para cada centro de la malla, el método busca cuál muestra está más cerca y le asigna **exactamente su valor**. No calcula un promedio ni utiliza las demás concentraciones para esa predicción. Repite la búsqueda para cada centro de forma independiente.

Con coordenadas planas en metros, comparamos las distancias euclidianas:

$$d=\sqrt{(x_{centro}-x_{muestra})^2+(y_{centro}-y_{muestra})^2}.$$

**Procedimiento:** calcular las distancias a las muestras → elegir la menor → copiar el zinc de esa muestra al centro de la malla.

### Un ejemplo que podemos revisar a mano

Imagina tres muestras en un sistema local de coordenadas métricas: A en (0, 0) con 100 mg/kg, B en (4, 0) con 400 mg/kg y C en (0, 3) con 900 mg/kg. Queremos estimar el valor en (1, 1).

Las distancias son aproximadamente 1.41 m a A, 3.16 m a B y 2.24 m a C. Como A es la más cercana, la predicción es **100 mg/kg**. El valor alto de C no interviene en ese resultado. Estas coordenadas son solo del ejemplo, no del conjunto Meuse.


In [ ]:
ejemplo_vecinos <- data.frame(
  muestra = c("A", "B", "C"), x_m = c(0, 4, 0), y_m = c(0, 0, 3),
  zinc_mg_kg = c(100, 400, 900)
)
consulta_ejemplo <- c(x_m = 1, y_m = 1)
ejemplo_vecinos$distancia_m <- sqrt(
  (ejemplo_vecinos$x_m - consulta_ejemplo["x_m"])^2 +
  (ejemplo_vecinos$y_m - consulta_ejemplo["y_m"])^2
)
ejemplo_vecinos
indice_vecino <- which.min(ejemplo_vecinos$distancia_m)
ejemplo_vecinos[indice_vecino, c("muestra", "zinc_mg_kg")]


`which.min()` devuelve la posición de la menor distancia. En este ejemplo no hay empate. Si dos muestras son equidistantes, debe revisarse la regla de desempate de la implementación; la distancia por sí sola no permite elegir una única muestra.

### ¿Por qué el mapa tiene parches y bordes bruscos?

Todos los lugares que comparten una misma muestra más cercana reciben el mismo valor. Así aparecen regiones constantes, equivalentes a zonas de proximidad o **Voronoi**. Al cruzar un borde cambia la muestra elegida y la concentración puede saltar de un valor a otro: no se crea una transición suave.

La malla representa esas regiones mediante cuadrados. Hacerla más fina dibuja sus bordes con más detalle, pero no agrega valores intermedios: la regla sigue siendo copiar una sola medición. Los puntos negros de los mapas son las muestras que originan esas regiones.

### Implementación con `gstat`

Usamos `idw()` con `nmax = 1`, de modo que solo participa el vecino más próximo. Con un único vecino, su peso se cancela al dividir por la suma de pesos: `idp` no cambia el valor predicho. `log_zinc ~ 1` indica la variable usada sin otra tendencia en la fórmula. El resultado guarda la predicción en `var1.pred`.

Aquí copiamos `log(zinc)` y después aplicamos `exp()`: obtenemos de nuevo el zinc de la muestra elegida. En cambio, el IDW de la siguiente sección combina varias muestras y sus pesos sí importan.

Es un buen método de referencia por su sencillez y porque conserva valores observados. Puede extender una medición aislada a una zona grande y no reconoce barreras como un cauce o un cambio de material. Sus bordes son consecuencia de la proximidad de las muestras, no evidencia de límites reales del suelo. Fuera del área muestreada seguirá copiando algún vecino, aunque esté muy lejos.


In [ ]:
pred_vecino <- idw(log_zinc ~ 1, locations = puntos, newdata = malla_sf,
                   nmax = 1, idp = 2, debug.level = 0)
malla$vecino <- exp(pred_vecino$var1.pred)


## 2. Distancia inversa ponderada (IDW)

$$\hat z(s_0)=\frac{\sum_i w_i z(s_i)}{\sum_i w_i},\qquad w_i=\frac{1}{d(s_i,s_0)^p}.$$

En esta fórmula, `z` será `log(zinc)` y `d` una distancia en metros. Compararemos potencias 1 y 2 con los 12 vecinos más próximos. Aumentar `p` refuerza la influencia de los puntos cercanos, pero no garantiza menor error. Estos valores son elecciones de la demostración, no parámetros óptimos demostrados.


In [ ]:
pred_idw1 <- idw(log_zinc ~ 1, locations = puntos, newdata = malla_sf,
                 idp = 1, nmax = 12, debug.level = 0)
pred_idw2 <- idw(log_zinc ~ 1, locations = puntos, newdata = malla_sf,
                 idp = 2, nmax = 12, debug.level = 0)
malla$idw_p1 <- exp(pred_idw1$var1.pred)
malla$idw_p2 <- exp(pred_idw2$var1.pred)


## 3. Una tendencia espacial plana

`lm(log_zinc ~ x_km + y_km)` ajusta una superficie de primer grado. Centramos las coordenadas y las expresamos en km para interpretar mejor sus coeficientes. El ajuste representa un patrón general; no es un interpolador exacto y puede dejar sin representar variaciones locales.

Usamos el mismo centro de coordenadas al preparar muestras y malla. `predict()` calcula el valor de la superficie en cada destino.


In [ ]:
centro_x <- mean(datos$x)
centro_y <- mean(datos$y)
datos$x_km <- (datos$x - centro_x) / 1000
datos$y_km <- (datos$y - centro_y) / 1000
malla$x_km <- (malla$x - centro_x) / 1000
malla$y_km <- (malla$y - centro_y) / 1000
tendencia <- lm(log_zinc ~ x_km + y_km, data = datos)
malla$tendencia <- exp(predict(tendencia, newdata = malla))
coef(tendencia)


## 4. Kriging ordinario: observar la dependencia espacial

El **variograma experimental** resume diferencias entre pares de mediciones según su separación. `variogram()` agrupa pares por intervalos de distancia. Usamos `cutoff = 1500` m y `width = 100` m: solo estos pares intervienen en este resumen.

Un variograma creciente sugiere que las mediciones próximas tienden a parecerse más. Esto debe revisarse junto a posibles tendencias y direcciones: un gráfico por sí solo no demuestra que se cumplan los supuestos.


In [ ]:
variograma <- variogram(log_zinc ~ 1, data = puntos, cutoff = 1500, width = 100)
plot(variograma, main = "Variograma experimental de log(Zn)")


### Ajustar un modelo de variograma

Usamos un modelo esférico (`"Sph"`). El **efecto pepita** representa variación a muy corta distancia y/o error de medición; la **meseta parcial** es la contribución espacial estructurada; el **alcance** marca la distancia a la que el modelo esférico llega a su meseta.

Los valores de `vgm()` son un punto de partida para el ajuste, no resultados. Inspecciona modelo y puntos. Si el ajuste es singular, detén la interpretación y revisa el modelo con el docente; no adoptes los parámetros automáticamente.


In [ ]:
modelo_inicial <- vgm(psill = 0.6, model = "Sph", range = 900, nugget = 0.05)
modelo <- fit.variogram(variograma, modelo_inicial)
modelo
stopifnot(!isTRUE(attr(modelo, "singular")))
plot(variograma, modelo, main = "Variograma experimental y modelo esférico")


### Predecir y conservar la varianza

`krige()` usa el modelo para predecir `log_zinc`. `var1.pred` es la predicción y `var1.var` su varianza bajo el modelo. Esta varianza no incluye toda la incertidumbre de seleccionar y ajustar el variograma, ni garantiza representar procesos omitidos.


In [ ]:
pred_kriging <- krige(log_zinc ~ 1, locations = puntos, newdata = malla_sf,
                      model = modelo, nmax = 12, debug.level = 0)
malla$kriging <- exp(pred_kriging$var1.pred)
malla$varianza_log <- pred_kriging$var1.var
summary(malla[, c("vecino", "idw_p1", "idw_p2", "tendencia", "kriging")])


## Mapas comparables

Transformamos los resultados a formato largo: cada fila es una celda y un método. Recorremos los nombres de los métodos con `for`, extraemos cada columna con `[[nombre]]` y reunimos las filas con `rbind()`. Todas las facetas usarán el mismo dominio, orientación, tamaño de celda y escala de color en mg/kg. Así las diferencias de color corresponden a diferencias de predicción.

Los puntos negros son ubicaciones medidas; las celdas son estimaciones. Los cuadrados junto al borde pueden sobresalir medio paso de malla porque la selección se hizo por sus centros. La escala es común y puede comprimir el contraste de algún método.

Usaremos la misma extensión elegida para la malla como ventana de todos los mapas. Al reducirla se ocultan los puntos exteriores de la figura, pero no se retiran del ajuste.


In [ ]:
metodos <- c("vecino", "idw_p1", "idw_p2", "tendencia", "kriging")
mapas <- data.frame()
for (nombre in metodos) {
  resultado_metodo <- data.frame(x = malla$x, y = malla$y,
                                 metodo = nombre, zinc_estimado = malla[[nombre]])
  mapas <- rbind(mapas, resultado_metodo)
}
mapas$metodo <- factor(mapas$metodo, levels = metodos,
                      labels = c("Vecino más cercano", "IDW p = 1", "IDW p = 2", "Tendencia plana", "Kriging ordinario"))
limites_color <- range(mapas$zinc_estimado, finite = TRUE)
figura_mapas <- ggplot(mapas, aes(x = x, y = y, fill = zinc_estimado)) +
  geom_tile(width = paso_m, height = paso_m) +
  geom_point(data = datos, aes(x = x, y = y), inherit.aes = FALSE,
             size = 0.35, color = "black") +
  facet_wrap(~ metodo, ncol = 3) +
  coord_equal(xlim = extension$x, ylim = extension$y, expand = FALSE) +
  scale_x_continuous(n.breaks = 3) +
  scale_fill_viridis_c(limits = limites_color, name = "Zn estimado
(mg/kg)") +
  labs(title = "De mediciones puntuales a superficies estimadas",
       subtitle = "Predicciones en log(Zn) devueltas con exp(); misma escala de color",
       x = "Este RD (m)", y = "Norte RD (m)") +
  theme_minimal(base_size = 10)
figura_mapas


## Cada modelación por separado

Además de la comparación conjunta, mostraremos los cinco mapas uno por uno. Conservamos la misma extensión, tamaño de celda y escala de color para poder compararlos. Las diferencias no se deben a que cada figura cambie automáticamente su leyenda.

La función siguiente reúne el diseño común. Recibe el método, la tabla de predicciones, las muestras, los límites de color, el paso y la extensión. Dentro, selecciona solo las filas de ese método. Cada llamada devuelve un gráfico independiente que podemos mostrar o guardar.


In [ ]:
dibujar_mapa <- function(nombre_metodo, tabla, muestras, limites, paso, extension) {
  seleccion <- tabla[tabla$metodo == nombre_metodo, ]
  figura <- ggplot(seleccion, aes(x = x, y = y, fill = zinc_estimado)) +
    geom_tile(width = paso, height = paso) +
    geom_point(data = muestras, aes(x = x, y = y), inherit.aes = FALSE,
               size = 0.7, color = "black") +
    coord_equal(xlim = extension$x, ylim = extension$y, expand = FALSE) +
    scale_x_continuous(n.breaks = 3) +
    scale_fill_viridis_c(limits = limites, name = "Zn estimado\n(mg/kg)") +
    labs(title = nombre_metodo, subtitle = "Puntos negros: ubicaciones medidas",
         x = "Este RD (m)", y = "Norte RD (m)") +
    theme_minimal()
  return(figura)
}


### Vecino más cercano


In [ ]:
figura_vecino <- dibujar_mapa("Vecino más cercano", mapas, datos, limites_color, paso_m, extension)
figura_vecino


### IDW p = 1


In [ ]:
figura_idw1 <- dibujar_mapa("IDW p = 1", mapas, datos, limites_color, paso_m, extension)
figura_idw1


### IDW p = 2


In [ ]:
figura_idw2 <- dibujar_mapa("IDW p = 2", mapas, datos, limites_color, paso_m, extension)
figura_idw2


### Tendencia plana


In [ ]:
figura_tendencia <- dibujar_mapa("Tendencia plana", mapas, datos, limites_color, paso_m, extension)
figura_tendencia


### Kriging ordinario


In [ ]:
figura_kriging <- dibujar_mapa("Kriging ordinario", mapas, datos, limites_color, paso_m, extension)
figura_kriging


### Un mapa de incertidumbre condicionada al modelo

Representamos la desviación estándar del kriging en escala logarítmica, calculada como la raíz de la varianza. **No está en mg/kg**. Se incluye una tolerancia numérica al llevar varianzas a cero; una varianza claramente negativa detendría el cálculo.


In [ ]:
stopifnot(all(malla$varianza_log > -1e-8))
malla$sd_log <- sqrt(pmax(malla$varianza_log, 0))
ggplot(malla, aes(x = x, y = y, fill = sd_log)) +
  geom_tile(width = paso_m, height = paso_m) +
  geom_point(data = datos, aes(x = x, y = y), inherit.aes = FALSE, size = 0.5) +
  coord_equal(xlim = extension$x, ylim = extension$y, expand = FALSE) +
  scale_fill_viridis_c(name = "DE de
log(Zn)") +
  labs(title = "Incertidumbre del kriging bajo el modelo ajustado",
       x = "Este RD (m)", y = "Norte RD (m)") +
  theme_minimal()


## Comprobar predicciones en ubicaciones reservadas

Una figura suave no demuestra precisión. Reservamos aproximadamente 20 % de las muestras y ajustamos cada método con el resto. El variograma se vuelve a estimar **solo con entrenamiento**; reutilizar el ajuste obtenido con todas las respuestas contaminaría esta comprobación.

Los parámetros IDW se fijaron antes de mirar el error. Esta partición aleatoria puede dejar puntos de prueba cerca de entrenamiento y dar una visión optimista de la transferencia a zonas sin muestras. Una evaluación más exigente usaría bloques espaciales y varias particiones. No ajustaremos de nuevo los métodos a partir de estos resultados.


In [ ]:
set.seed(2026)
indice_prueba <- sample(seq_len(nrow(datos)), size = floor(0.2 * nrow(datos)))
entrenamiento <- puntos[-indice_prueba, ]
prueba <- puntos[indice_prueba, ]
c(entrenamiento = nrow(entrenamiento), prueba = nrow(prueba))


In [ ]:
valid_vecino <- idw(log_zinc ~ 1, entrenamiento, prueba, nmax = 1, debug.level = 0)
valid_idw1 <- idw(log_zinc ~ 1, entrenamiento, prueba, idp = 1, nmax = 12, debug.level = 0)
valid_idw2 <- idw(log_zinc ~ 1, entrenamiento, prueba, idp = 2, nmax = 12, debug.level = 0)
tendencia_valid <- lm(log_zinc ~ x_km + y_km, data = datos[-indice_prueba, ])
vario_valid <- variogram(log_zinc ~ 1, data = entrenamiento, cutoff = 1500, width = 100)
modelo_valid <- fit.variogram(vario_valid, modelo_inicial)
stopifnot(!isTRUE(attr(modelo_valid, "singular")))
valid_kriging <- krige(log_zinc ~ 1, entrenamiento, prueba,
                       model = modelo_valid, nmax = 12, debug.level = 0)


### Resumir errores

El **MAE** es el error absoluto medio; el **RMSE** penaliza más los errores grandes; el **sesgo** es el promedio de predicción menos observación. Los calcularemos en mg/kg, después de `exp()`. Un menor error describe esta partición y esta variable, no una superioridad universal.

Como no aplicamos una corrección de transformación para predecir medias aritméticas, la vuelta desde logaritmos puede contribuir al sesgo en mg/kg. Conserva ese límite al interpretar la tabla.


In [ ]:
predicciones_valid <- data.frame(
  vecino = exp(valid_vecino$var1.pred),
  idw_p1 = exp(valid_idw1$var1.pred),
  idw_p2 = exp(valid_idw2$var1.pred),
  tendencia = exp(predict(tendencia_valid, newdata = datos[indice_prueba, ])),
  kriging = exp(valid_kriging$var1.pred)
)
observado <- datos$zinc[indice_prueba]
errores <- data.frame()
for (nombre in names(predicciones_valid)) {
  error <- predicciones_valid[[nombre]] - observado
  resultado_error <- data.frame(
    metodo = nombre, n = length(error), MAE_mg_kg = mean(abs(error)),
    RMSE_mg_kg = sqrt(mean(error^2)), sesgo_mg_kg = mean(error)
  )
  errores <- rbind(errores, resultado_error)
}
errores[order(errores$RMSE_mg_kg), ]


In [ ]:
dir.create("figuras", showWarnings = FALSE)
ggsave("figuras/comparacion_interpolacion.png", plot = figura_mapas,
       width = 11, height = 8, units = "in", dpi = 200)


También guardamos cada mapa individual en su propio PNG. Esta lista reúne los cinco objetos gráficos; el ciclo usa su nombre para construir el archivo correspondiente.


In [ ]:
figuras_individuales <- list(vecino = figura_vecino, idw_p1 = figura_idw1,
                             idw_p2 = figura_idw2, tendencia = figura_tendencia,
                             kriging = figura_kriging)
for (nombre in names(figuras_individuales)) {
  ggsave(file.path("figuras", paste0("interpolacion_", nombre, ".png")),
         plot = figuras_individuales[[nombre]], width = 7, height = 7,
         units = "in", dpi = 200)
}


## Para discutir con los mapas a la vista

- ¿Qué métodos generan bordes abruptos y cuáles suavizan más?
- ¿Qué cambia entre las dos potencias IDW? ¿Dónde hay poco apoyo de las muestras?
- ¿Coinciden el mejor aspecto visual y el menor error en prueba?
- ¿Qué cambiaría al interpolar otra variable, reducir la densidad de muestreo o reservar una región completa?

Los mapas estiman concentraciones para este conjunto y estos supuestos. No constituyen una clasificación de riesgo ambiental: eso requeriría información y criterios adicionales.

## Documentación para consultar

[IDW y kriging en gstat](https://r-spatial.github.io/gstat/reference/krige.html) · [Variograma experimental](https://r-spatial.github.io/gstat/reference/variogram.html) · [Ajuste de variogramas](https://r-spatial.github.io/gstat/reference/fit.variogram.html).

